In [ ]:
!pip -q install folium plotly kaleido pyarrow
!pip -q install branca
import folium, plotly
import pyarrow
import kaleido
print("folium:", folium.__version__)
print("plotly:", plotly.__version__)
print("pyarrow:", pyarrow.__version__)

In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
PARQUET_PATH = Path(r"C:\Users\anton\Projects\Diana\BigData\traffic_full.parquet")  # change to actual path
assert PARQUET_PATH.exists(), f"Parquet not found: {PARQUET_PATH}"
EXPORT_ROOT = Path("assets")
(EXPORT_ROOT / "maps").mkdir(parents=True, exist_ok=True)
(EXPORT_ROOT / "charts").mkdir(parents=True, exist_ok=True)
(EXPORT_ROOT / "charts_png").mkdir(parents=True, exist_ok=True)
(EXPORT_ROOT / "data").mkdir(parents=True, exist_ok=True)
(EXPORT_ROOT / "data" / "factor_tables").mkdir(parents=True, exist_ok=True)
PARQUET_PATH, EXPORT_ROOT.resolve()

In [ ]:
import pyarrow.parquet as pq
try:
    pf = pq.ParquetDataset(str(PARQUET_PATH))
    schema_cols = [name for name in pf.schema.names]
except Exception as e:
    print(f"ParquetDataset failed: {e}. Trying direct read for schema...")
    df_test = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
    schema_cols = df_test.columns.tolist()
    del df_test
print(f"Columns in parquet: {len(schema_cols)}")
schema_cols[:30]

In [ ]:
CANDIDATE_COLS = [
    "latitude", "longitude", "speed_limit",
    "casualty_severity",
    "weather_conditions", "road_surface_conditions", "light_conditions",
    "road_type", "first_road_class", "second_road_class",
    "accident_date", "date", "accident_datetime", "datetime", "accident_time", "time",
    "accident_year", "year", "accident_month", "month", "accident_day", "day",
    "lsoa_of_accident_location", "lsoa_of_casualty",
    "local_authority_district", "police_force",
 ]
VIZ_COLS = [c for c in CANDIDATE_COLS if c in schema_cols]
missing = [c for c in CANDIDATE_COLS if c not in schema_cols]
print("Using columns:", VIZ_COLS)
print("Missing (ok):", missing[:20], "..." if len(missing) > 20 else "")

In [ ]:
try:
    df = pd.read_parquet(PARQUET_PATH, columns=VIZ_COLS if len(VIZ_COLS) > 0 else None, engine="pyarrow")
except Exception as e:
    print(f"Read with column selection failed: {e}. Reading all columns...")
    df = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
    if VIZ_COLS:
        df = df[[c for c in VIZ_COLS if c in df.columns]]

if "__null_dask_index__" in df.columns:
    df = df.drop(columns=["__null_dask_index__"])

for col in ["latitude", "longitude", "speed_limit", "casualty_severity"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
if {"latitude", "longitude"}.issubset(df.columns):
    df = df.dropna(subset=["latitude", "longitude"])
    df = df[(df["latitude"].between(49, 61)) & (df["longitude"].between(-9, 3))]
if "casualty_severity" in df.columns:
    df = df.dropna(subset=["casualty_severity"]).copy()
    df["casualty_severity"] = df["casualty_severity"].astype(int)
after = len(df)
print(f"Rows: {before:,} → {after:,} after basic filters")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

In [ ]:
key_cols = [c for c in ["latitude","longitude","speed_limit","casualty_severity","weather_conditions","light_conditions"] if c in df.columns]
missing_rate = (df[key_cols].isna().mean() * 100).round(2) if key_cols else pd.Series(dtype=float)
print("Missing-rate (%):")
display(missing_rate.to_frame("missing_%"))

In [ ]:
severity_map = {1: "Fatal", 2: "Serious", 3: "Slight"}
if "casualty_severity" in df.columns:
    df["severity_code"] = df["casualty_severity"].astype("Int64")
    df["severity_label"] = df["severity_code"].map(severity_map)
    df["severity_weight"] = df["severity_code"].map({1: 3, 2: 2, 3: 1})
else:
    df["severity_code"] = pd.NA
    df["severity_label"] = pd.NA
    df["severity_weight"] = pd.NA

def build_datetime(df: pd.DataFrame) -> pd.Series:
    for col in ["datetime", "accident_datetime"]:
        if col in df.columns:
            return pd.to_datetime(df[col], errors="coerce")
    date_col = next((c for c in ["accident_date", "date"] if c in df.columns), None)
    time_col = next((c for c in ["accident_time", "time"] if c in df.columns), None)
    if date_col and time_col:
        dt = pd.to_datetime(df[date_col].astype(str) + " " + df[time_col].astype(str), errors="coerce")
        return dt
    year_col = next((c for c in ["accident_year", "year"] if c in df.columns), None)
    month_col = next((c for c in ["accident_month", "month"] if c in df.columns), None)
    day_col = next((c for c in ["accident_day", "day"] if c in df.columns), None)
    if year_col and month_col:
        day_vals = df[day_col] if day_col else 1
        return pd.to_datetime(dict(year=df[year_col], month=df[month_col], day=day_vals), errors="coerce")
    return pd.Series([pd.NaT] * len(df), index=df.index)

df["dt"] = build_datetime(df)
df["year"] = df["dt"].dt.year
df["month"] = df["dt"].dt.month
df["day_of_week"] = df["dt"].dt.day_name()
df["hour"] = df["dt"].dt.hour

display(df[["latitude","longitude","severity_label","severity_weight","dt","year","month","day_of_week","hour"]].head())
print(df["severity_label"].value_counts(dropna=False))

In [ ]:
VIZ_KEEP = [c for c in [
    "latitude", "longitude", "severity_code", "severity_label", "severity_weight",
    "dt", "year", "month", "day_of_week", "hour",
    "weather_conditions", "road_surface_conditions", "light_conditions",
    "road_type", "first_road_class", "second_road_class",
    "speed_limit", "local_authority_district", "police_force",
    "lsoa_of_accident_location", "lsoa_of_casualty",
] if c in df.columns]
df_viz = df[VIZ_KEEP].copy()
print("df_viz columns:", df_viz.columns.tolist())
print(f"df_viz rows: {len(df_viz):,}")

SAMPLE_MAP_MAX = 200_000
if len(df_viz) > SAMPLE_MAP_MAX:
    df_viz_map = df_viz.sample(SAMPLE_MAP_MAX, random_state=42)
else:
    df_viz_map = df_viz.copy()

cache_sample_path = EXPORT_ROOT / "data" / "df_viz_sample.parquet"
df_viz_map.to_parquet(cache_sample_path, index=False)
print("Saved sample:", cache_sample_path)
print(f"Sample rows: {len(df_viz_map):,}")
print(f"Sample memory: {df_viz_map.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
from folium.plugins import MarkerCluster, FastMarkerCluster, HeatMap
from folium import FeatureGroup

def add_scale_control(m):
    if hasattr(folium, "ScaleControl"):
        folium.ScaleControl().add_to(m)
        return
    try:
        from folium.plugins import ScaleBar
        ScaleBar().add_to(m)
    except Exception:
        pass

if {"latitude","longitude"}.issubset(df_viz_map.columns):
    center_lat = df_viz_map["latitude"].median()
    center_lon = df_viz_map["longitude"].median()
else:
    center_lat, center_lon = 54.0, -2.0

m_hotspots = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

cluster_group = FeatureGroup(name="MarkerCluster")
marker_cluster = MarkerCluster()
color_map = {"Fatal": "red", "Serious": "orange", "Slight": "blue"}

rows_for_markers = df_viz_map.dropna(subset=["latitude","longitude"]).head(50_000)
for _, r in rows_for_markers.iterrows():
    sev = r.get("severity_label", None)
    color = color_map.get(sev, "gray")
    popup_txt = f"{sev or 'Unknown'}"
    if pd.notna(r.get("dt", pd.NaT)):
        popup_txt += f" | {r['dt']}"
    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=2, color=color, fill=True, fill_opacity=0.6, popup=popup_txt
    ).add_to(marker_cluster)

marker_cluster.add_to(cluster_group)
cluster_group.add_to(m_hotspots)

fast_group = FeatureGroup(name="FastMarkerCluster")
fast_points = df_viz_map[["latitude", "longitude"]].dropna().values.tolist()
FastMarkerCluster(fast_points).add_to(fast_group)
fast_group.add_to(m_hotspots)

folium.LayerControl().add_to(m_hotspots)
add_scale_control(m_hotspots)
m_hotspots

In [ ]:
m_heat = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

heat_basic = FeatureGroup(name="HeatMap (all)" )
heat_weighted = FeatureGroup(name="HeatMap (severity-weighted)")

basic_points = df_viz_map[["latitude","longitude"]].dropna().values.tolist()
HeatMap(basic_points, radius=8, blur=6, min_opacity=0.2).add_to(heat_basic)
heat_basic.add_to(m_heat)

if "severity_weight" in df_viz_map.columns:
    weighted_points = df_viz_map[["latitude","longitude","severity_weight"]].dropna().values.tolist()
    HeatMap(weighted_points, radius=10, blur=8, min_opacity=0.25).add_to(heat_weighted)
    heat_weighted.add_to(m_heat)

folium.LayerControl().add_to(m_heat)
add_scale_control(m_heat)
m_heat

In [ ]:
grid_decimals = 2 # ~1km grid cells
df_grid = df_viz.copy()
df_grid = df_grid.dropna(subset=["latitude","longitude"])
df_grid["lat_cell"] = df_grid["latitude"].round(grid_decimals)
df_grid["lon_cell"] = df_grid["longitude"].round(grid_decimals)

grp = (df_grid.groupby(["lat_cell","lon_cell","severity_label"]).size()
       .reset_index(name="count"))
pivot = grp.pivot(index=["lat_cell","lon_cell"], columns="severity_label", values="count").fillna(0)
pivot.columns = [str(c) for c in pivot.columns]
pivot = pivot.reset_index()
for col in ["Slight", "Serious", "Fatal"]:
    if col not in pivot.columns:
        pivot[col] = 0

pivot["score"] = pivot["Slight"] + 2 * pivot["Serious"] + 3 * pivot["Fatal"]
pivot["accidents"] = pivot[["Slight","Serious","Fatal"]].sum(axis=1)
top_cells = pivot.sort_values("score", ascending=False).head(200)

hotspot_csv = EXPORT_ROOT / "data" / "hotspot_cells_top.csv"
top_cells.to_csv(hotspot_csv, index=False)
print("Saved:", hotspot_csv)
top_cells.head(10)

In [ ]:
m_hotspots_grid = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")
top_n = top_cells.head(100)
max_score = top_n["score"].max() if len(top_n) else 1
for _, r in top_n.iterrows():
    radius = 2 + 8 * (r["score"] / max_score)
    folium.CircleMarker(
        location=[r["lat_cell"], r["lon_cell"]],
        radius=radius, color="crimson", fill=True, fill_opacity=0.5,
        popup=f"Score: {r['score']} | Accidents: {r['accidents']}"
    ).add_to(m_hotspots_grid)

m_hotspots_grid

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

figs = {}
agg_tables = {}

df_time = df_viz.copy()
df_time = df_time.dropna(subset=["dt"])
if len(df_time) == 0:
    print("No datetime available for time charts.")
else:
    # By hour
    hour_counts = df_time.groupby("hour").size().reset_index(name="accidents")
    figs["by_hour"] = px.bar(hour_counts, x="hour", y="accidents", title="Accidents by Hour")
    agg_tables["by_hour"] = hour_counts

    # By day of week (ordered)
    dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    dow_counts = df_time.groupby("day_of_week").size().reset_index(name="accidents")
    dow_counts["day_of_week"] = pd.Categorical(dow_counts["day_of_week"], categories=dow_order, ordered=True)
    dow_counts = dow_counts.sort_values("day_of_week")
    figs["by_dayofweek"] = px.bar(dow_counts, x="day_of_week", y="accidents", title="Accidents by Day of Week")
    agg_tables["by_dayofweek"] = dow_counts

    # By month
    month_counts = df_time.groupby("month").size().reset_index(name="accidents")
    figs["by_month"] = px.bar(month_counts, x="month", y="accidents", title="Accidents by Month")
    agg_tables["by_month"] = month_counts

    # By year
    year_counts = df_time.groupby("year").size().reset_index(name="accidents")
    figs["by_year"] = px.line(year_counts, x="year", y="accidents", title="Accidents by Year")
    agg_tables["by_year"] = year_counts

list(figs.keys())

In [ ]:
if len(df_time) > 0 and "severity_label" in df_time.columns:
    yearly = df_time.groupby(["year","severity_label"]).size().reset_index(name="accidents")
    yearly_total = df_time.groupby("year").size().reset_index(name="accidents")
    figs["accidents_over_time"] = px.line(yearly_total, x="year", y="accidents", title="Accidents Over Time (All)" )

    yearly_pivot = yearly.pivot_table(index="year", columns="severity_label", values="accidents", fill_value=0).reset_index()
    yearly_pivot["total"] = yearly_pivot[[c for c in yearly_pivot.columns if c != "year"]].sum(axis=1)
    for c in ["Fatal","Serious","Slight"]:
        if c in yearly_pivot.columns:
            yearly_pivot[c] = (yearly_pivot[c] / yearly_pivot["total"]).fillna(0)
    fig_share = go.Figure()
    for c in ["Fatal","Serious","Slight"]:
        if c in yearly_pivot.columns:
            fig_share.add_trace(go.Scatter(x=yearly_pivot["year"], y=yearly_pivot[c], mode="lines", name=c))
    fig_share.update_layout(title="Yearly Severity Share", yaxis_title="Share")
    figs["severity_share"] = fig_share
    agg_tables["yearly_severity_share"] = yearly_pivot
else:
    print("Skipping yearly severity share (missing datetime or severity_label)")

In [ ]:
factor_figs = {}
factor_tables = {}

def factor_rate_table(df_in: pd.DataFrame, col: str, min_count: int = 1000):
    if col not in df_in.columns:
        return None
    dfc = df_in.dropna(subset=[col, "severity_label"]).copy()
    dfc["is_serious_or_fatal"] = dfc["severity_label"].isin(["Fatal","Serious"]).astype(int)
    dfc["is_fatal"] = (dfc["severity_label"] == "Fatal").astype(int)
    grouped = dfc.groupby(col).agg(
        n=("severity_label", "size"),
        serious_or_fatal_rate=("is_serious_or_fatal", "mean"),
        fatal_rate=("is_fatal", "mean")
    ).reset_index()
    grouped = grouped[grouped["n"] >= min_count]
    grouped["serious_or_fatal_rate"] = (grouped["serious_or_fatal_rate"] * 100).round(2)
    grouped["fatal_rate"] = (grouped["fatal_rate"] * 100).round(2)
    return grouped.sort_values("serious_or_fatal_rate", ascending=False)

df_factor = df_viz.copy()
if "speed_limit" in df_factor.columns:
    df_factor["speed_bin"] = pd.cut(df_factor["speed_limit"], bins=[0,20,30,40,50,60,70,80,100,140], right=True)

factor_cols = [c for c in ["weather_conditions","light_conditions","road_surface_conditions","road_type","speed_bin"] if c in df_factor.columns]
for col in factor_cols:
    tbl = factor_rate_table(df_factor, col, min_count=1000)
    if tbl is not None and len(tbl) > 0:
        factor_tables[col] = tbl
        factor_figs[col] = px.bar(tbl, x=col, y="serious_or_fatal_rate", title=f"Serious/Fatal Rate by {col}")

list(factor_figs.keys())

In [ ]:
exported = []

def export_folium(m, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(path))
    exported.append(path)
    return path

def export_plotly(fig, html_path: Path, png_path: Path | None = None):
    html_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    exported.append(html_path)
    if png_path is not None:
        try:
            png_path.parent.mkdir(parents=True, exist_ok=True)
            fig.write_image(str(png_path), scale=2)
            exported.append(png_path)
        except Exception as e:
            print(f"PNG export failed for {png_path.name}: {e}")
    return html_path

def save_table(df_table: pd.DataFrame, csv_path: Path, parquet_path: Path | None = None):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_table.to_csv(csv_path, index=False)
    exported.append(csv_path)
    if parquet_path is not None:
        df_table.to_parquet(parquet_path, index=False)
        exported.append(parquet_path)
    return csv_path

# Export maps
export_folium(m_hotspots, EXPORT_ROOT / "maps" / "accidents_cluster_map.html")
export_folium(m_heat, EXPORT_ROOT / "maps" / "accidents_heatmap.html")
export_folium(m_hotspots_grid, EXPORT_ROOT / "maps" / "accidents_hotspot_grid.html")

# Export charts
for key, fig in figs.items():
    export_plotly(fig, EXPORT_ROOT / "charts" / f"{key}.html", EXPORT_ROOT / "charts_png" / f"{key}.png")

# Export aggregated tables
for key, tbl in agg_tables.items():
    save_table(tbl, EXPORT_ROOT / "data" / f"{key}.csv")

# Export factor tables
for key, tbl in factor_tables.items():
    safe_key = re.sub(r"[^a-zA-Z0-9_]+", "_", key)
    save_table(tbl, EXPORT_ROOT / "data" / "factor_tables" / f"factor_{safe_key}.csv")

# Export hotspot cells table (already saved in Section 7)
exported.append(hotspot_csv)

print("Export manifest:")
for p in exported:
    print("-", p.as_posix())